# 02 — Mixture-of-Experts from first principles, and trained from scratch

**Assignment, part 2.** Train the same model but with MoE instead of the linear (MLP) layers.

Before training anything we open up the MoE layer: how a token is routed, why renormalising the gates matters,
what the load-balancing loss does, and that the fast sorted dispatch equals a naive per-token loop.
Then we train an MoE with *exactly* the dense run's budget (same data order, steps, learning rate) as the
from-scratch reference for notebook 03.

## The layer

For a token representation $x\in\mathbb R^d$ and $E$ experts $f_e$ (each an MLP identical in shape to the dense one):

$$p = \mathrm{softmax}(W_r x),\qquad S=\mathrm{TopK}(p,k),\qquad g_e=\frac{p_e}{\sum_{j\in S}p_j}\ (e\in S),\qquad y=\sum_{e\in S} g_e\,f_e(x).$$

* Only $k$ of $E$ experts run per token → **parameters grow ×E in the MoE layers but compute only ×k**.
* Gates are **renormalised over the chosen experts** (they sum to 1). If all $f_e$ are the same function $f$, then $y=f(x)$ —
  this is what will make the dense → MoE conversion lossless.
* **Load-balancing loss** (Switch Transformer): with $f_e$ = fraction of routed slots sent to expert $e$ and $P_e$ = mean router
  probability for $e$,  $\mathcal L_{lb}=E\sum_e f_e P_e$. It equals 1 when the load is perfectly uniform and grows when a few experts hog tokens.
* **Router z-loss**: $\mathbb E[\log\sum_e e^{z_e}]^2$ keeps router logits small (numerical stability).
* Dispatch is *dropless*: no capacity limit, no dropped tokens.

In [ ]:
import sys, os, math, time, json, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, os.environ.get("MOE_SRC", os.path.abspath("../src")))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from moe.config import get_config
from moe.utils import set_seed, get_device, savefig, plot_style, aim_repo, work_root, ckpt_dir, results_dir, is_smoke
from moe.data import load_dataset_tokens
from moe.model import GPT, GPTConfig, MoEMLP, param_report
from moe.runs import dense_config, moe_config, train_config, flops_per_token
from moe.trainer import train, load_history, load_model, lr_at
from moe.upcycle import upcycle, max_logit_diff
from moe.viz import ema, color, heatmap_load
from moe.analysis import make_decoder, make_encoder

plot_style()
X = get_config()
DEVICE = get_device()
torch.set_float32_matmul_precision("high")
set_seed(X.seed)
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)

print(f"torch {torch.__version__} | device = {DEVICE}"
      + (f" ({torch.cuda.get_device_name(0)}, {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB)" if DEVICE == "cuda" else ""))
print(f"preset = {X.name!r}   (override with MOE_PRESET=smoke|small|base)")
print(f"Aim repo : {aim_repo()}   -> browse with:  uv run aim up")
print()
print(pd.Series(X.to_dict(), name="value").to_string())

## 1. Watching the router work on a handful of tokens

In [ ]:
set_seed(0)
toy_cfg = GPTConfig(vocab_size=100, block_size=8, n_layer=1, n_head=1, n_embd=16, n_experts=4, top_k=2, moe_every=1)
layer = MoEMLP(toy_cfg); torch.nn.init.normal_(layer.router.weight, std=0.5)   # a opinionated router for the demo
x = torch.randn(1, 6, 16)
probs = layer.router(x.reshape(-1, 16)).softmax(-1)
topv, topi = probs.topk(2, -1)
gates = topv / topv.sum(-1, keepdim=True)
rows = []
for t in range(6):
    rows.append({"token": t, **{f"p[e{e}]": round(probs[t, e].item(), 3) for e in range(4)},
                 "top-2": topi[t].tolist(), "raw top-2 p": [round(v, 3) for v in topv[t].tolist()],
                 "renormalised gates": [round(v, 3) for v in gates[t].tolist()],
                 "gate sum": round(gates[t].sum().item(), 3)})
print(pd.DataFrame(rows).to_string(index=False))
y, aux = layer(x)
print("\nlayer output shape:", tuple(y.shape), "| per-expert load:", [round(v, 3) for v in aux["load"].tolist()],
      "| lb loss:", round(aux["lb"].item(), 3), "| z loss:", round(aux["z"].item(), 3))

## 2. Correctness: sorted dispatch vs naive per-token loop

In [ ]:
layer.eval()
x = torch.randn(3, 7, 16)
y_fast, _ = layer(x)
xf = x.reshape(-1, 16)
pr = layer.router(xf).softmax(-1); tv, ti = pr.topk(2, -1); tv = tv / tv.sum(-1, keepdim=True)
ref = torch.zeros_like(xf)
for t in range(xf.shape[0]):
    for j in range(2):
        e = ti[t, j]
        h = torch.nn.functional.gelu(torch.nn.functional.linear(xf[t], layer.w1[e], layer.b1[e]))
        ref[t] += tv[t, j] * torch.nn.functional.linear(h, layer.w2[e], layer.b2[e])
err = (y_fast.reshape(-1, 16) - ref).abs().max().item()
print(f"max |fast - naive| = {err:.2e}")
assert err < 1e-5

## 3. What does the load-balancing loss do? (toy experiment)

Tokens come from 4 Gaussian clusters with **skewed frequencies** (55 / 25 / 15 / 5 %). Each cluster has its own linear
target map, so experts have a reason to specialise. We train a single MoE layer with and without the balancing term and watch
how the routed load per expert evolves.

In [ ]:
def run_toy(lb_coef, steps=300, seed=0, E=4, d=16):
    g = torch.Generator().manual_seed(seed)
    centers = torch.randn(E, d, generator=g) * 3
    A = torch.randn(E, d, d, generator=g) / d ** 0.5
    freq = torch.tensor([0.55, 0.25, 0.15, 0.05])
    cfg = GPTConfig(vocab_size=8, block_size=8, n_layer=1, n_head=1, n_embd=d // 4 * 4, n_experts=E, top_k=2, moe_every=1)
    torch.manual_seed(seed)
    L = MoEMLP(cfg); opt = torch.optim.Adam(L.parameters(), lr=3e-3)
    loads, mses, ents = [], [], []
    for s in range(steps):
        c = torch.multinomial(freq, 256, replacement=True, generator=g)
        xb = centers[c] + torch.randn(256, d, generator=g)
        yb = torch.einsum("nij,nj->ni", A[c], xb)
        out, aux = L(xb[None])
        mse = ((out[0] - yb) ** 2).mean()
        (mse + lb_coef * aux["lb"]).backward(); opt.step(); opt.zero_grad()
        loads.append(aux["load"].tolist()); mses.append(mse.item()); ents.append(aux["entropy"].item())
    return np.array(loads), np.array(mses), np.array(ents)

toy = {lb: run_toy(lb) for lb in (0.0, 0.05)}
fig, axs = plt.subplots(1, 3, figsize=(16, 3.8))
for ax, (lb, (loads, mses, ents)) in zip(axs[:2], toy.items()):
    ax.stackplot(range(len(loads)), loads.T, labels=[f"expert {e}" for e in range(4)], alpha=0.85)
    ax.set_title(f"routed load per expert, lb_coef = {lb}"); ax.set_xlabel("step"); ax.set_ylim(0, 1); ax.legend(loc="upper right", fontsize=8)
for lb, (loads, mses, ents) in toy.items():
    axs[2].plot(ema(mses, 0.05), label=f"lb_coef={lb}")
axs[2].set_yscale("log"); axs[2].set_title("task MSE (EMA)"); axs[2].set_xlabel("step"); axs[2].legend()
plt.tight_layout(); savefig(fig, "02_toy_load_balance.png"); plt.show()
for lb, (loads, mses, ents) in toy.items():
    print(f"lb_coef={lb:<5} final load = {np.round(loads[-20:].mean(0), 3).tolist()} | "
          f"max-load {loads[-20:].mean(0).max():.2f} (fair share 0.25) | final MSE {mses[-20:].mean():.4f} | "
          f"router entropy {ents[-20:].mean():.3f} (ln4={math.log(4):.3f})")

## 4. Dense vs MoE: parameters and compute

In [ ]:
data = load_dataset_tokens(X.dataset)
dense_m = GPT(dense_config(X, data.vocab_size))
moe_m = GPT(moe_config(X, data.vocab_size))
rows = {}
for name, m in (("dense", dense_m), ("moe", moe_m)):
    rp = param_report(m)
    rows[name] = {"total params (M)": rp["total"] / 1e6, "active params / token (M)": rp["active"] / 1e6,
                  "MFLOPs / token": flops_per_token(m) / 1e6,
                  "MoE blocks": len(m.moe_blocks()), **{f"  {k} (M)": v / 1e6 for k, v in rp["groups"].items()}}
cmp = pd.DataFrame(rows).fillna(0)
print(cmp.round(3).to_string())
print(f"\nMoE stores {rows['moe']['total params (M)']/rows['dense']['total params (M)']:.2f}x the parameters of the dense model "
      f"but spends only {rows['moe']['MFLOPs / token']/rows['dense']['MFLOPs / token']:.2f}x the FLOPs per token.")
del dense_m, moe_m

## 5. Train the MoE from scratch (same budget as the dense run)

Same seed, same batches, same schedule as `dense` — the only difference is the architecture. Besides cross-entropy the loss now
includes `lb_coef · L_lb + z_coef · L_z`. The trainer logs per-layer expert load, load entropy and the number of 'dead' experts to Aim.

In [ ]:
mcfg = moe_config(X, data.vocab_size)
tc = train_config(X, "moe_scratch", "moe_scratch", X.dense_steps, X.dense_lr, X.dense_warmup, device=DEVICE)
set_seed(X.seed)
model = GPT(mcfg)
model, HM = train(model, data, tc, experiment="session14", aim_repo=aim_repo(),
                  extra_hparams={"arch": "moe", "init": "scratch"})

## 6. Dense vs MoE-from-scratch

In [ ]:
HM = load_history("moe_scratch")
try:
    HD = load_history("dense")
except FileNotFoundError:
    HD = None; print("(run notebook 01 to add the dense curves to these plots)")

fig, axs = plt.subplots(1, 3, figsize=(17, 4.2))
for lab, H in (("dense", HD), ("moe_scratch", HM)):
    if H is None: continue
    axs[0].plot(H["step"], ema(H["loss"], 0.1), color=color(lab), label=f"{lab} train (EMA)")
    axs[0].plot(H["eval_step"], H["val_loss"], "o--", color=color(lab), label=f"{lab} val")
axs[0].set_title("loss"); axs[0].set_xlabel("step"); axs[0].legend(); axs[0].set_ylim(top=min(8, max(HM["loss"])))
for lab, H in (("dense", HD), ("moe_scratch", HM)):
    if H is None: continue
    axs[1].plot(H["eval_step"], H["val_loss"], "o-", color=color(lab), label=lab)
axs[1].set_title("validation loss (zoom)"); axs[1].set_xlabel("step"); axs[1].legend()
lo = min(min(HM["val_loss"]), min(HD["val_loss"]) if HD else 99); axs[1].set_ylim(lo - 0.1, lo + 1.5)
for lab, H in (("dense", HD), ("moe_scratch", HM)):
    if H is None: continue
    axs[2].plot(H["step"][1:], np.array(H["tok_s"][1:]) / 1e3, color=color(lab), label=lab)
axs[2].set_title("throughput (k tok/s)"); axs[2].set_xlabel("step"); axs[2].legend()
plt.tight_layout(); savefig(fig, "02_dense_vs_moe_scratch.png"); plt.show()

rows = {}
for lab, H in (("dense", HD), ("moe_scratch", HM)):
    if H is None: continue
    rows[lab] = {"total params (M)": H["n_params"] / 1e6, "active (M)": H["n_active_params"] / 1e6,
                 "final val loss": H["final_val_loss"], "final val ppl": math.exp(H["final_val_loss"]),
                 "wall-clock (min)": H["wall_clock_s"] / 60, "tok/s (k)": np.mean(H["tok_s"][1:]) / 1e3,
                 "peak mem (GB)": H["peak_mem_gb"]}
print(pd.DataFrame(rows).round(3).to_string())

## 7. Routing behaviour during training

In [ ]:
load = np.array(HM["load"])           # [n_eval, L_moe, E]
E = load.shape[2]
fig, axs = plt.subplots(1, 3, figsize=(17, 4.2))
im = heatmap_load(axs[0], load[-1], f"expert load at the end (fair share = {1/E:.3f})")
plt.colorbar(im, ax=axs[0])
ent = np.array(HM["load_entropy"])
for li in range(ent.shape[1]): axs[1].plot(HM["eval_step"], ent[:, li], label=f"MoE layer {li}")
axs[1].set_title("load entropy (1 = perfectly balanced)"); axs[1].set_xlabel("step"); axs[1].legend(); axs[1].set_ylim(0, 1.02)
axs[2].plot(HM["step"], HM["lb"], color="#d95f02"); axs[2].axhline(1, color="k", lw=0.6, ls="--")
axs[2].set_title("load-balancing loss (1.0 = uniform)"); axs[2].set_xlabel("step")
plt.tight_layout(); savefig(fig, "02_moe_scratch_routing.png"); plt.show()
print("final load per expert, per MoE layer:"); print(pd.DataFrame(load[-1]).round(3).to_string())
print("dead experts at the end:", HM["dead_experts"][-1])